## Your pipeline, part 3: fusion of the two final models
HW2, notebook 9 (**Kaggle CPU** is enough, no GPU quota; a GPU is used automatically if one is on).

**Why.** The two final pipelines are both YOLO11s + `A2+A3_wbf` inference, but trained differently:
`final_04` (960 px, 8 epochs, leaderboard 0.2694 via notebook 07) and `final_08` (1280 px, 20 epochs, leaderboard 0.2393).
08 is better on the official val (0.2067 vs 0.2045) but worse on test, so the two models make different errors.
Averaging the boxes of models with different errors is the most reliable post-hoc gain (Weighted Boxes Fusion across models,
Solovyev et al., arXiv:1910.13302). Each fused box takes at most one box per model, so neighbouring objects of one model
(dense traffic) are never merged together. Expected gain: modest, about +0.005 to +0.015 mAP; not guaranteed.

**What is fused.** For each image, the merged output of each model (at most 500 boxes, after its own six-pass WBF):
- test: the two submission CSVs already made (07 for `final_04`, 08 for `final_08`), so **no test prediction is needed**;
- val: 08's saved val passes, and `final_04`'s six val passes, computed here once (the slow part, ~1-1.5 h on CPU) and saved
  as `passes_final_04_val.pkl.gz` (notebook 08's ensemble mode can reuse it).

**Fusion rule** (one WBF across the two models): boxes of one class are clustered at IoU > `iou`; a fused box is the
score x weight average of its members; its score is the weighted mean score of the models that found it, times
(weight share of those models) ^ `p`. So `p = 0` keeps a box found by one model at full score, `p = 1` halves it (equal weights),
`p = 2` quarters it. Inside 08, a stronger penalty for boxes seen by fewer passes was better (power 1-2 beat 0).

**Search on val, small and staged** (val has only 7 clips; a large grid would fit val noise, which already misled once):
1. `p` in {0, 1, 2} at equal weights, IoU 0.55;
2. with the best `p`: weight of `final_04` in {0.6, 0.7} (it generalised better to the test clips);
3. with the best of those: IoU in {0.5, 0.6}.

**Go / no-go.** Submit only if the best fusion beats `final_04` alone on val by at least **+0.01** and also beats `final_08` alone.
The notebook writes the CSV in either case (for the report) and prints the verdict.

**Inputs:** the competition data; the outputs of notebooks **04** (`final_04.pt`), **07** (`submission_07_A2_A3_wbf.csv`,
`results_07_inference.csv`) and **08** (`submission_08_A2_A3_wbf.csv`, `passes_08_val.pkl.gz`, `results_08_summary.csv`).
**Internet ON** (pip). Runtime: ~1.5-2 h on CPU, ~30-45 min with a GPU.

In [1]:
%pip install -q ultralytics

In [2]:
import os, io, json, time, math, random, shutil, contextlib, gc, gzip, pickle
from pathlib import Path
from multiprocessing import Pool
import numpy as np, pandas as pd, cv2, torch
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.ops import batched_nms
from ultralytics import YOLO, settings
from ultralytics.utils import DEFAULT_CFG_DICT
settings.update({"wandb": False})

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

GPU = torch.cuda.is_available()
DEVICE_ID = 0 if GPU else "cpu"
PREC = ({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True}) if GPU else {}   # fp16 on GPU only
print("device:", torch.cuda.get_device_name(0) if GPU else f"CPU, {os.cpu_count()} cores")

# ---- same inference settings as notebooks 07 / 08
VAL_STEP = 5
UP_SCALE, TILE_ZOOM, TILE_OVERLAP = 1.33, 1.5, 0.2
NMS_IOU, WBF_IOU, MAX_DET = 0.6, 0.55, 500
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]
MERGE_WORKERS = max(1, (os.cpu_count() or 2))          # CPU processes for WBF / fusion
GO_MARGIN = 0.01                                       # required val gain over final_04 alone

# ---- inputs: fail here, not after an hour
TEST_CSV = find("test_images.csv"); ROOT = TEST_CSV.parent
VAL_DIR = find("val/annotations_coco.json").parent
need = {"final_04.pt": "notebook 04", "submission_07_A2_A3_wbf.csv": "notebook 07", "results_07_inference.csv": "notebook 07",
        "submission_08_A2_A3_wbf.csv": "notebook 08", "passes_08_val.pkl.gz": "notebook 08", "results_08_summary.csv": "notebook 08"}
missing = [f"{f} ({src})" for f, src in need.items() if not find_all(f)]
assert not missing, "attach the outputs that contain: " + ", ".join(missing)
for f in need:
    print(f"{f:30s} {find(f)}")

device: Tesla T4
final_04.pt                    /kaggle/input/notebooks/vivtrea/04-yolo11/final_04.pt
submission_07_A2_A3_wbf.csv    /kaggle/input/notebooks/vivtrea/pipeline-inference/submission_07_A2_A3_wbf.csv
results_07_inference.csv       /kaggle/input/notebooks/vivtrea/pipeline-inference/results_07_inference.csv
submission_08_A2_A3_wbf.csv    /kaggle/input/notebooks/vivtrea/pipeline-training/submission_08_A2_A3_wbf.csv
passes_08_val.pkl.gz           /kaggle/input/notebooks/vivtrea/pipeline-training/passes_08_val.pkl.gz
results_08_summary.csv         /kaggle/input/notebooks/vivtrea/pipeline-training/results_08_summary.csv


### Data and evaluator (same as notebooks 02-08)

In [3]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [4]:
# official val: every VAL_STEP-th frame of each clip, exactly the frames of notebooks 07 / 08
VAL_GT = load_gt(VAL_DIR / "annotations_coco.json")
_idx = {(p.parent.name, p.name): p for p in (VAL_DIR / "images").rglob("*.jpg")}
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_ITEMS = [(int(r.id), _idx[(r.video, Path(r.file_name).name)]) for r in _v.itertuples()]
VAL_IDS = [i for i, _ in VAL_ITEMS]
TEST_IDS = list(pd.read_csv(TEST_CSV).image_id)
print(f"{len(VAL_ITEMS)} val frames | {len(TEST_IDS)} test frames")

# reference numbers of the two pipelines on val (from their own notebooks)
REF_04 = float(pd.read_csv(find("results_07_inference.csv")).set_index("strategy").loc["A2+A3_wbf", "mAP"])
REF_08 = float(pd.read_csv(find("results_08_summary.csv")).set_index("config").loc["final_08 A2+A3_wbf", "mAP"])
print(f"val mAP reported by notebook 07 (final_04): {REF_04:.4f} | by notebook 08 (final_08): {REF_08:.4f}")

571 val frames | 3811 test frames
val mAP reported by notebook 07 (final_04): 0.2045 | by notebook 08 (final_08): 0.2067


### Prediction passes and merging (code of notebooks 07 / 08)

In [5]:
def load_model(path):
    # model + the image size it was trained at
    ck = torch.load(path, map_location="cpu", weights_only=False)
    imgsz = int(np.max(ck.get("train_args", {}).get("imgsz", 960)))
    del ck
    return YOLO(str(path)), imgsz

def run(model, images, imgsz):
    # one model call on a few images (Ultralytics 8.4 puts a whole list into one batch); one (N, 6) array per image
    res = model.predict(images, imgsz=imgsz, conf=0.001, iou=0.6, max_det=MAX_DET, verbose=False, device=DEVICE_ID, **PREC)
    return [np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()].astype(np.float32) for r in res]

def unflip(b, width):
    # boxes predicted on a horizontally flipped image -> original orientation
    b = b.copy(); b[:, [0, 2]] = width - b[:, [2, 0]]
    return b

def nms(b, iou):
    # class-wise NMS, best first, at most MAX_DET boxes (main process only)
    if not len(b):
        return b
    t = torch.from_numpy(b)
    keep = batched_nms(t[:, :4], t[:, 4], t[:, 5].long(), iou).numpy()
    return b[keep[:MAX_DET]]

def tile_grid(W, H):
    # overlapping crops of side max(W, H) / TILE_ZOOM (clipped to the frame), at least TILE_OVERLAP overlap
    T = int(max(W, H) / TILE_ZOOM)
    tw, th = min(T, W), min(T, H)
    def starts(size, t):
        n = max(1, math.ceil((size - t) / (t * (1 - TILE_OVERLAP))) + 1) if size > t else 1
        return np.linspace(0, size - t, n).round().astype(int)
    return [(x, y, x + tw, y + th) for y in starts(H, th) for x in starts(W, tw)]

ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def predict_views(model, imgsz, img):
    # all six passes of one frame, boxes in original pixel coordinates
    H, W = img.shape[:2]
    up = int(round(imgsz * UP_SCALE / 32) * 32)
    flip = np.ascontiguousarray(img[:, ::-1])
    out = {"full": run(model, [img], imgsz)[0], "full_flip": unflip(run(model, [flip], imgsz)[0], W),
           "up": run(model, [img], up)[0], "up_flip": unflip(run(model, [flip], up)[0], W)}
    grid = tile_grid(W, H)
    for name in ("tiles", "tiles_flip"):
        crops = [img[y0:y1, x0:x1] for x0, y0, x1, y1 in grid]
        if name == "tiles_flip":
            crops = [np.ascontiguousarray(c[:, ::-1]) for c in crops]
        parts = []
        for (x0, y0, x1, y1), b in zip(grid, run(model, crops, imgsz)):
            if name == "tiles_flip":
                b = unflip(b, x1 - x0)
            # drop boxes cut by an inner crop edge: the neighbouring crop (>= 20 % overlap) or the full pass has the whole object
            e = 2
            cut = (((b[:, 0] <= e) & (x0 > 0)) | ((b[:, 2] >= x1 - x0 - e) & (x1 < W)) |
                   ((b[:, 1] <= e) & (y0 > 0)) | ((b[:, 3] >= y1 - y0 - e) & (y1 < H)))
            b = b[~cut].copy(); b[:, [0, 2]] += x0; b[:, [1, 3]] += y0
            parts.append(b)
        out[name] = nms(np.concatenate(parts), NMS_IOU)       # remove duplicates from overlapping crops
    return out

def passes(model_path, tag, items):
    # prediction passes of one model on a set of frames; reused from this or an attached output, else computed and saved
    fname = f"passes_{tag}.pkl.gz"
    hit = [p for p in [WORK / fname] if p.exists()] + find_all(fname)
    if hit:
        with gzip.open(hit[0], "rb") as f:
            cache = pickle.load(f)
        if set(cache) == {i for i, _ in items}:
            print(f"reusing {hit[0]}")
            return cache
        print(f"{hit[0]} covers other frames, recomputing")
    model, imgsz = load_model(model_path)
    print(f"{tag}: {Path(model_path).name} trained at {imgsz} px, {len(items)} frames")
    cache = {iid: predict_views(model, imgsz, cv2.imread(str(p))) for iid, p in tqdm(items, desc=tag)}
    del model; gc.collect(); torch.cuda.empty_cache()
    with gzip.open(WORK / fname, "wb") as f:
        pickle.dump(cache, f, protocol=4)
    return cache

In [6]:
def wbf(views, iou_thr=WBF_IOU):
    # views: list of (N, 6) arrays, one per pass; returns fused (M, 6), best first, at most MAX_DET
    n_views = len(views)
    allb = np.concatenate([v for v in views if len(v)] or [np.zeros((0, 6), np.float32)])
    out = []
    for c in np.unique(allb[:, 5]):
        b = allb[allb[:, 5] == c]
        b = b[np.argsort(-b[:, 4])]
        acc = np.zeros((len(b), 4)); fb = np.zeros((len(b), 4)); w = np.zeros(len(b)); cnt = np.zeros(len(b)); m = 0
        for x1, y1, x2, y2, s, _ in b:
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += s * np.array([x1, y1, x2, y2]); w[j] += s; cnt[j] += 1
                    fb[j] = acc[j] / w[j]
                    continue
            acc[m] = s * np.array([x1, y1, x2, y2]); w[m] = s; cnt[m] = 1; fb[m] = (x1, y1, x2, y2); m += 1
        score = w[:m] / cnt[:m] * np.minimum(cnt[:m], n_views) / n_views
        out.append(np.c_[fb[:m], score, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    res = np.concatenate(out).astype(np.float32)
    return res[np.argsort(-res[:, 4])][:MAX_DET]

def merge(views, how):
    if how is None:                                         # single pass: as predicted
        return views[0][:MAX_DET]
    if how == "nms":
        return nms(np.concatenate(views), NMS_IOU)
    return wbf(views)

STRATEGIES = {                                              # name: (views, merge)
    "S0_single_pass":   (["full"], None),
    "A1_upscaled_pass": (["up"], None),
    "A2_tta_nms":       (["full", "full_flip", "up", "up_flip"], "nms"),
    "A2_tta_wbf":       (["full", "full_flip", "up", "up_flip"], "wbf"),
    "A3_tiles_nms":     (["full", "tiles"], "nms"),
    "A3_tiles_wbf":     (["full", "tiles"], "wbf"),
    "A2+A3_wbf":        (["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"], "wbf"),
}
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def merge_task(args):
    # one image, one strategy (runs in a worker process)
    iid, views, how = args
    t0 = time.perf_counter()
    b = merge(views, how)
    return iid, b, 1000 * (time.perf_counter() - t0)

def merge_all(cache, views, how):
    # merge the cached passes of every frame; returns detections (COCO style) and merge ms per frame
    tasks = [(iid, [cache[iid][v] for v in views], how) for iid in cache]
    # worker processes only for WBF (pure NumPy). NMS uses PyTorch, which can hang inside forked processes
    # once its thread pool was used in the main process; NMS takes only seconds here anyway.
    if MERGE_WORKERS and how == "wbf":
        with Pool(MERGE_WORKERS) as p:
            res = p.map(merge_task, tasks, chunksize=8)
    else:
        res = [merge_task(t) for t in tasks]
    dets = [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b, _ in res for r in b]
    return dets, float(np.mean([t for _, _, t in res]))

### Val: each model alone
`final_04`'s six passes are computed here (the slow step); the merged result must reproduce notebook 07's number, which checks
that the right model and the same frames are used. 08's passes are loaded from its output.

In [7]:
def merged(cache, ids):
    # six-pass WBF of each image (strategy A2+A3_wbf), rounded like the submission CSVs (score 4, box 1 decimal)
    tasks = [(iid, [cache[iid][v] for v in ALL_VIEWS], "wbf") for iid in ids]
    with Pool(MERGE_WORKERS) as p:
        res = p.map(merge_task, tasks, chunksize=8)
    return {iid: csv_round(b) for iid, b, _ in res}

def csv_round(b):
    # same precision as a submission row: the val numbers then match what the test CSVs contain
    b = b.copy().astype(np.float64)
    wh = np.round(b[:, 2:4] - b[:, 0:2], 1); b[:, 0:2] = np.round(b[:, 0:2], 1); b[:, 2:4] = b[:, 0:2] + wh
    b[:, 4] = np.round(b[:, 4], 4)
    return b.astype(np.float32)

def to_dets(by):
    return [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b in by.items() for r in b]

t0 = time.time()
cache04 = passes(find("final_04.pt"), "final_04_val", VAL_ITEMS)       # reused if an earlier run's output is attached
print(f"final_04 val passes: {(time.time() - t0) / 60:.1f} min")
with gzip.open(find("passes_08_val.pkl.gz"), "rb") as f:
    cache08 = pickle.load(f)
assert set(cache08) == set(VAL_IDS), "passes_08_val.pkl.gz covers other val frames than this notebook"

A_val, B_val = merged(cache04, VAL_IDS), merged(cache08, VAL_IDS)
del cache04, cache08; gc.collect()
rows = [{"variant": "final_04 alone (A2+A3_wbf)", **coco_eval(VAL_GT, to_dets(A_val), VAL_IDS)},
        {"variant": "final_08 alone (A2+A3_wbf)", **coco_eval(VAL_GT, to_dets(B_val), VAL_IDS)}]
display(pd.DataFrame(rows).set_index("variant").round(4))
# a wrong model or other frames would differ by far more (CPU fp32 vs GPU fp16 changes the 4th decimal)
assert abs(rows[0]["mAP"] - REF_04) < 0.005, f"final_04 gives {rows[0]['mAP']:.4f} on val, notebook 07 reported {REF_04:.4f}"
assert abs(rows[1]["mAP"] - REF_08) < 0.005, f"final_08 gives {rows[1]['mAP']:.4f} on val, notebook 08 reported {REF_08:.4f}"
SOLO_04, SOLO_08 = rows[0]["mAP"], rows[1]["mAP"]

final_04_val: final_04.pt trained at 960 px, 571 frames
final_04 val passes: 2.4 min


,mAP,AP50,AP_s,AP_m,AP_l
variant,,,,,
final_04 alone (A2+A3_wbf),0.2046,0.4058,0.1645,0.2059,0.3278
final_08 alone (A2+A3_wbf),0.2066,0.4159,0.1288,0.2127,0.3080


### Fusion of the two models

In [8]:
def fuse(a, b, w_a, iou_thr, p):
    # WBF across two models' merged outputs; a, b: (N, 6) [x1, y1, x2, y2, score, class]; returns (M, 6), best first
    w = np.array([w_a, 1.0 - w_a])
    allb = np.concatenate([np.c_[a, np.zeros(len(a))], np.c_[b, np.ones(len(b))]]) if len(a) + len(b) else np.zeros((0, 7))
    out = []
    for c in np.unique(allb[:, 5]):
        x = allb[allb[:, 5] == c]
        x = x[np.argsort(-x[:, 4] * w[x[:, 6].astype(int)], kind="stable")]   # best weighted score first
        n = len(x)
        acc = np.zeros((n, 4)); fb = np.zeros((n, 4)); sw = np.zeros(n); best = np.zeros((n, 2)); m = 0
        for x1, y1, x2, y2, s, _, k in x:
            k = int(k); ws = s * w[k]
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                iou[best[:m, k] > 0] = 0                               # at most one box per model in a fused box
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += ws * np.array([x1, y1, x2, y2]); sw[j] += ws; fb[j] = acc[j] / sw[j]
                    best[j, k] = s
                    continue
            acc[m] = ws * np.array([x1, y1, x2, y2]); sw[m] = ws; fb[m] = (x1, y1, x2, y2); best[m] = 0; best[m, k] = s; m += 1
        found = best[:m] > 0
        share = (found * w).sum(1)                                     # weight share of the models that found the box
        mean = (best[:m] * w).sum(1) / share                           # weighted mean score of those models
        out.append(np.c_[fb[:m], mean * share ** p, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    r = np.concatenate(out).astype(np.float32)
    return r[np.argsort(-r[:, 4])][:MAX_DET]

def fuse_task(args):
    iid, a, b, w_a, iou_thr, p = args
    return iid, fuse(a, b, w_a, iou_thr, p)

def fuse_all(A, B, ids, w_a, iou_thr, p):
    empty = np.zeros((0, 6), np.float32)
    with Pool(MERGE_WORKERS) as pool:
        res = pool.map(fuse_task, [(i, A.get(i, empty), B.get(i, empty), w_a, iou_thr, p) for i in ids], chunksize=16)
    return dict(res)

def try_variant(w_a, iou_thr, p):
    name = f"fusion w04={w_a} iou={iou_thr} p={p}"
    m = coco_eval(VAL_GT, to_dets(fuse_all(A_val, B_val, VAL_IDS, w_a, iou_thr, p)), VAL_IDS)
    rows.append({"variant": name, **m, "w04": w_a, "iou": iou_thr, "p": p})
    print(f"{name:34s} mAP {m['mAP']:.4f} | AP_s {m['AP_s']:.4f} | vs final_04 alone {m['mAP'] - SOLO_04:+.4f}", flush=True)
    return m["mAP"]

# stage 1: how to score boxes found by one model only
s1 = {p: try_variant(0.5, 0.55, p) for p in (0, 1, 2)}
P = max(s1, key=s1.get)
# stage 2: weight of final_04 (better on the test clips)
s2 = {0.5: s1[P], **{w: try_variant(w, 0.55, P) for w in (0.6, 0.7)}}
W04 = max(s2, key=s2.get)
# stage 3: matching IoU between the two models' boxes
s3 = {0.55: s2[W04], **{t: try_variant(W04, t, P) for t in (0.5, 0.6)}}
IOU = max(s3, key=s3.get)
best_map = s3[IOU]
res_val = pd.DataFrame(rows).set_index("variant")
res_val.to_csv(WORK / "results_09_val.csv")
display(res_val.round(4))
print(f"best fusion: w04={W04} iou={IOU} p={P} -> val mAP {best_map:.4f}")

fusion w04=0.5 iou=0.55 p=0        mAP 0.2159 | AP_s 0.1461 | vs final_04 alone +0.0113
fusion w04=0.5 iou=0.55 p=1        mAP 0.2230 | AP_s 0.1501 | vs final_04 alone +0.0183
fusion w04=0.5 iou=0.55 p=2        mAP 0.2248 | AP_s 0.1494 | vs final_04 alone +0.0202
fusion w04=0.6 iou=0.55 p=2        mAP 0.2238 | AP_s 0.1509 | vs final_04 alone +0.0192
fusion w04=0.7 iou=0.55 p=2        mAP 0.2216 | AP_s 0.1590 | vs final_04 alone +0.0170
fusion w04=0.5 iou=0.5 p=2         mAP 0.2248 | AP_s 0.1493 | vs final_04 alone +0.0201
fusion w04=0.5 iou=0.6 p=2         mAP 0.2250 | AP_s 0.1493 | vs final_04 alone +0.0204


,mAP,AP50,AP_s,AP_m,AP_l,w04,iou,p
variant,,,,,,,,
final_04 alone (A2+A3_wbf),0.2046,0.4058,0.1645,0.2059,0.3278,NaN,NaN,NaN
final_08 alone (A2+A3_wbf),0.2066,0.4159,0.1288,0.2127,0.3080,NaN,NaN,NaN
fusion w04=0.5 iou=0.55 p=0,0.2159,0.4257,0.1461,0.2173,0.3330,0.5,0.55,0.0
fusion w04=0.5 iou=0.55 p=1,0.2230,0.4405,0.1501,0.2247,0.3376,0.5,0.55,1.0
fusion w04=0.5 iou=0.55 p=2,0.2248,0.4447,0.1494,0.2272,0.3388,0.5,0.55,2.0
fusion w04=0.6 iou=0.55 p=2,0.2238,0.4417,0.1509,0.2255,0.3400,0.6,0.55,2.0
fusion w04=0.7 iou=0.55 p=2,0.2216,0.4373,0.1590,0.2225,0.3404,0.7,0.55,2.0
fusion w04=0.5 iou=0.5 p=2,0.2248,0.4448,0.1493,0.2270,0.3387,0.5,0.50,2.0
fusion w04=0.5 iou=0.6 p=2,0.2250,0.4451,0.1493,0.2272,0.3389,0.5,0.60,2.0


best fusion: w04=0.5 iou=0.6 p=2 -> val mAP 0.2250


**Observations:**
- Penalising boxes found by only one model matters most: p = 0 (keep them at full score) +0.011 over final_04 alone, p = 1 +0.018, p = 2 +0.020. A box confirmed by both models is much more likely to be a true object.
- Equal weights are best for mAP (0.225). A higher weight for final_04 lowers mAP (0.224 at 0.6, 0.222 at 0.7) but raises AP_small (0.151 -> 0.159), because final_04 + tiles is better on small objects.
- The matching IoU (0.5 / 0.55 / 0.6) barely matters (±0.0002).
- Best: +0.020 over final_04 and +0.018 over final_08 on val. AP_small (0.149) stays below final_04 alone (0.164).

### Test submission and verdict

In [9]:
def read_submission(path):
    # submission CSV -> {image_id: (N, 6) [x1, y1, x2, y2, score, class]}
    out = {}
    for iid, s in pd.read_csv(path).fillna("").itertuples(index=False):
        v = np.array(s.split(), dtype=np.float64).reshape(-1, 6)       # class score x y w h
        out[iid] = np.c_[v[:, 2], v[:, 3], v[:, 2] + v[:, 4], v[:, 3] + v[:, 5], v[:, 1], v[:, 0]].astype(np.float32)
    return out

A_test, B_test = read_submission(find("submission_07_A2_A3_wbf.csv")), read_submission(find("submission_08_A2_A3_wbf.csv"))
assert set(A_test) == set(B_test) == set(TEST_IDS), "the two submission CSVs cover other test images than test_images.csv"
fused = fuse_all(A_test, B_test, TEST_IDS, W04, IOU, P)
write_submission(to_dets(fused), WORK / "submission_09_fusion.csv")

gain = best_map - SOLO_04
go = gain >= GO_MARGIN and best_map > SOLO_08
print(f"\nval mAP: final_04 {SOLO_04:.4f} | final_08 {SOLO_08:.4f} | fusion {best_map:.4f} ({gain:+.4f} vs final_04)")
print("VERDICT:", "GO -> submit submission_09_fusion.csv" if go else
      f"NO-GO -> gain below +{GO_MARGIN} on val: keep notebook 07's submission (0.2694); do not spend a submission")

saved /kaggle/working/submission_09_fusion.csv: 3811 rows, 3811 with detections, 1828594 boxes
val mAP: final_04 0.2046 | final_08 0.2066 | fusion 0.2250 (+0.0204 vs final_04)
VERDICT: GO -> submit submission_09_fusion.csv


**Observations:**
- GO: +0.020 on val. Leaderboard 0.2765 vs 0.2694 for final_04 alone (+0.007), so about a third of the val gain transferred.
- Fusing two models made from the same YOLO11s helps even though one of them (08) was worse on the leaderboard: their errors differ (different resolution and schedule), and the fusion keeps mainly the boxes both agree on.